# QUE Outlier Detection

This notebook applies **Quantum Entropy (QUE) scoring** and several baseline outlier detection methods to a user-supplied DataFrame. It is adapted directly from the code in the [QUE paper repository](https://github.com/twistedcubic/que-outlier-detection).

Each method assigns a **score per row** — higher scores indicate a higher likelihood of being an outlier.

### Methods implemented
| Name | Description |
|---|---|
| `que` | QUE scoring: matrix exponential of the covariance, parameterised by α |
| `naive_spectral` | Project onto the top eigenvector of the covariance |
| `l2` | Euclidean distance to the empirical mean |
| `knn` | Mean distance to the k nearest neighbours |
| `lof` | Local outlier factor (reachability-distance variant) |
| `isolation_forest` | scikit-learn IsolationForest |
| `elliptic_envelope` | scikit-learn EllipticEnvelope |

### Quick-start
1. Install requirements (cell below).
2. Load your DataFrame in the **"Load your data"** cell.
3. Adjust `NUMERIC_COLS`, `ALPHA`, and other settings in the **Config** cell.
4. Run all cells.

In [ ]:
# ── Requirements ──────────────────────────────────────────────────────────────
# Uncomment and run once if packages are not already installed.
# !pip install torch numpy pandas scikit-learn scipy matplotlib seaborn

In [ ]:
import torch
import numpy as np
import numpy.linalg as linalg
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.ensemble
import sklearn.covariance
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Config
Adjust these before running.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────

# Columns to use as features. Set to None to use all numeric columns.
NUMERIC_COLS = None

# QUE alpha parameter. Controls entropy of U:
#   alpha=0  → reduces to l2 norm
#   alpha→∞  → reduces to naive spectral scoring
#   alpha=4 is a reliable rule-of-thumb from the paper.
ALPHA = 4.0

# Number of nearest neighbours for kNN and LOF methods.
K_NEIGHBORS = 10

# Expected fraction of outliers. Used by EllipticEnvelope.
CONTAMINATION = 0.1

# Whether to standardise features (zero mean, unit variance) before scoring.
# Recommended when features are on very different scales.
STANDARDISE = True

# Methods to run. Remove any you do not need.
METHODS = ['que', 'naive_spectral', 'l2', 'knn', 'lof', 'isolation_forest', 'elliptic_envelope']

## Load your data
Replace the example below with your own DataFrame.

In [ ]:
# ── Load your data ────────────────────────────────────────────────────────────
# Replace this with your own loading code, e.g.:
#   df = pd.read_csv('my_data.csv')

# --- Example: synthetic dataset with planted outliers ---
np.random.seed(42)
n_inliers  = 500
n_outliers = 50
n_features = 20
n_corrupt_dirs = 4          # number of directions outliers bias

inliers  = np.random.randn(n_inliers,  n_features)
outliers = np.random.randn(n_outliers, n_features) * 0.1
chunk = n_outliers // n_corrupt_dirs
for i in range(n_corrupt_dirs):
    bias = np.sqrt(n_corrupt_dirs / 0.09)   # matches paper's synthetic scaling
    sign = 1 if i % 2 == 0 else -1
    outliers[i*chunk:(i+1)*chunk, i] += sign * bias

X_all   = np.vstack([inliers, outliers])
is_outlier = np.array([0]*n_inliers + [1]*n_outliers)  # ground-truth labels

df = pd.DataFrame(X_all, columns=[f'feature_{i}' for i in range(n_features)])
df['is_outlier'] = is_outlier   # ground-truth column (optional)

print(f'DataFrame shape: {df.shape}')
df.head()

## Preprocessing

In [ ]:
# ── Feature extraction ────────────────────────────────────────────────────────
if NUMERIC_COLS is None:
    # Automatically detect numeric columns; skip any ground-truth / label column
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # Drop 'is_outlier' if present — it's ground truth, not a feature
    feature_cols = [c for c in feature_cols if c != 'is_outlier']
else:
    feature_cols = NUMERIC_COLS

print(f'Using {len(feature_cols)} feature columns: {feature_cols[:6]}{"..." if len(feature_cols)>6 else ""}')

X_np = df[feature_cols].values.astype(np.float32)

if STANDARDISE:
    col_mean = X_np.mean(axis=0, keepdims=True)
    col_std  = X_np.std(axis=0, keepdims=True)
    col_std[col_std == 0] = 1.0     # avoid divide-by-zero for constant columns
    X_np = (X_np - col_mean) / col_std
    print('Features standardised.')

# Centre for covariance-based methods
X_centered = X_np - X_np.mean(axis=0, keepdims=True)

X_t  = torch.from_numpy(X_centered).to(device)
print(f'Feature matrix shape: {X_t.shape}')

## Core scoring functions
All scoring logic is implemented here — no external module dependencies.

In [ ]:
# ── Covariance helper ─────────────────────────────────────────────────────────
def compute_cov(X: torch.Tensor) -> torch.Tensor:
    """Empirical covariance of X (already centred)."""
    return torch.mm(X.t(), X) / X.size(0)


# ── QUE scoring ───────────────────────────────────────────────────────────────
def que_score(X: torch.Tensor, alpha: float = 4.0) -> torch.Tensor:
    """
    Quantum Entropy (QUE) outlier scores (Algorithm 2 in the paper).

    U  = exp(alpha * Sigma / ||Sigma||_2) / tr(exp(alpha * Sigma / ||Sigma||_2))
    tau_i = (X_i - mu)^T U (X_i - mu)

    Parameters
    ----------
    X     : (n, d) tensor, already centred.
    alpha : scaling parameter (paper default: 4).

    Returns
    -------
    tau   : (n,) tensor of outlier scores. Higher => more likely outlier.
    """
    Sigma = compute_cov(X)                     # (d, d)
    Sigma_np = Sigma.cpu().numpy().astype(np.float64)
    spectral_norm = float(np.linalg.norm(Sigma_np, ord=2))

    # SVD of alpha * Sigma / ||Sigma||_2
    scaled = alpha * Sigma_np / max(spectral_norm, 1e-10)
    U_svd, D, _ = linalg.svd(scaled)

    # U_matrix = V diag(exp(D)) V^T
    U_svd_t = torch.from_numpy(U_svd.astype(np.float32)).to(device)
    D_exp   = torch.from_numpy(np.exp(D.astype(np.float32))).to(device).diag()

    U_matrix = torch.mm(U_svd_t, torch.mm(D_exp, U_svd_t.t()))  # (d, d)
    U_matrix = U_matrix / U_matrix.diag().sum()                   # normalise

    # tau_i = x_i^T U x_i
    X_U = torch.mm(X, U_matrix)                # (n, d)
    tau = (X * X_U).sum(dim=-1)               # (n,)
    return tau


# ── Naive spectral scoring ────────────────────────────────────────────────────
def naive_spectral_score(X: torch.Tensor) -> torch.Tensor:
    """
    Project each point onto the top eigenvector of the empirical covariance.
    tau_i = <X_i, v>^2 where v is the leading eigenvector.
    """
    Sigma = compute_cov(X).cpu().numpy().astype(np.float64)
    U_svd, _, _ = linalg.svd(Sigma)
    v = torch.from_numpy(U_svd[:, 0].astype(np.float32)).to(device)  # (d,)
    tau = torch.mv(X, v) ** 2
    return tau


# ── L2 scoring ────────────────────────────────────────────────────────────────
def l2_score(X: torch.Tensor) -> torch.Tensor:
    """Squared Euclidean distance from the empirical mean."""
    return (X ** 2).sum(dim=-1)


# ── Pairwise L2 distance helper ───────────────────────────────────────────────
def pairwise_l2(X: torch.Tensor) -> torch.Tensor:
    """(n, n) matrix of squared L2 distances."""
    norms = (X ** 2).sum(dim=1, keepdim=True)   # (n, 1)
    dist  = norms + norms.t() - 2 * torch.mm(X, X.t())
    dist  = dist.clamp(min=0.0)                  # numerical safety
    return dist


# ── kNN scoring ───────────────────────────────────────────────────────────────
def knn_score(X: torch.Tensor, k: int = 10) -> torch.Tensor:
    """
    Mean distance to the k nearest neighbours.
    Excludes the point itself.
    """
    dist = pairwise_l2(X)                        # (n, n)
    # fill diagonal with large value so point is not its own neighbour
    dist.fill_diagonal_(float('inf'))
    k_eff = min(k, X.size(0) - 1)
    top_k, _ = torch.topk(dist, k=k_eff, largest=False, dim=-1)
    return top_k.mean(dim=-1)


# ── LOF scoring ───────────────────────────────────────────────────────────────
def lof_score(X: torch.Tensor, k: int = 10) -> torch.Tensor:
    """
    Local Outlier Factor using reachability distance (local method).
    Adapted directly from baselines.py in the repo.
    Higher scores indicate outliers.
    """
    dist = pairwise_l2(X)                        # (n, n)
    dist.fill_diagonal_(float('inf'))
    k_eff = min(k, X.size(0) - 1)

    min_dist, min_idx = torch.topk(dist, k=k_eff, largest=False, dim=-1)  # (n, k)
    # kth distance for each point
    kth_dist = min_dist[:, -1]                   # (n,)

    # Reachability distance: max(kth_dist[neighbour], actual_dist)
    kth_dist_exp = kth_dist.unsqueeze(0).expand(X.size(0), -1)           # (n, n)
    kth_dist_nbr = torch.gather(kth_dist_exp, dim=1, index=min_idx)      # (n, k)
    reach_dist   = torch.max(min_dist, kth_dist_nbr)                     # (n, k)

    lrd = reach_dist.mean(dim=-1).clamp(min=1e-6)  # local reachability density
    return lrd   # higher lrd → point is in a denser region → lower LOF, so
                 # returning lrd directly means higher = more isolated (outlier-like)


# ── Isolation Forest ──────────────────────────────────────────────────────────
def isolation_forest_score(X: torch.Tensor) -> torch.Tensor:
    """scikit-learn IsolationForest. Higher score = more likely outlier."""
    X_np = X.cpu().numpy()
    model = sklearn.ensemble.IsolationForest(contamination='auto', random_state=42)
    model.fit(X_np)
    scores = -model.decision_function(X_np)   # flip sign: higher = worse
    return torch.from_numpy(scores.astype(np.float32)).to(device)


# ── Elliptic Envelope ─────────────────────────────────────────────────────────
def elliptic_envelope_score(X: torch.Tensor, contamination: float = 0.1) -> torch.Tensor:
    """scikit-learn EllipticEnvelope (Mahalanobis-distance based). Higher = outlier."""
    X_np = X.cpu().numpy()
    model = sklearn.covariance.EllipticEnvelope(contamination=contamination)
    model.fit(X_np)
    scores = -model.decision_function(X_np)
    return torch.from_numpy(scores.astype(np.float32)).to(device)


print('All scoring functions defined.')

## Run all methods

In [ ]:
# ── Run scoring ───────────────────────────────────────────────────────────────
scores = {}

for method in METHODS:
    print(f'Running {method}...', end=' ')
    try:
        if method == 'que':
            s = que_score(X_t, alpha=ALPHA)
        elif method == 'naive_spectral':
            s = naive_spectral_score(X_t)
        elif method == 'l2':
            s = l2_score(X_t)
        elif method == 'knn':
            s = knn_score(X_t, k=K_NEIGHBORS)
        elif method == 'lof':
            s = lof_score(X_t, k=K_NEIGHBORS)
        elif method == 'isolation_forest':
            s = isolation_forest_score(X_t)
        elif method == 'elliptic_envelope':
            s = elliptic_envelope_score(X_t, contamination=CONTAMINATION)
        else:
            print(f'Unknown method "{method}", skipping.')
            continue

        scores[method] = s.cpu().numpy()
        print('done.')
    except Exception as e:
        print(f'FAILED ({e})')

# Attach scores to the original DataFrame
scores_df = df.copy()
for method, s in scores.items():
    scores_df[f'score_{method}'] = s

score_cols = [f'score_{m}' for m in scores]
print(f'\nScore columns added: {score_cols}')
scores_df[score_cols].describe()

## Results — Top outlier rows

In [ ]:
# ── Top 20 rows by QUE score ───────────────────────────────────────────────────
if 'score_que' in scores_df.columns:
    top20 = scores_df.nlargest(20, 'score_que')[['score_' + m for m in scores] + (['is_outlier'] if 'is_outlier' in scores_df else [])]
    print('Top 20 rows by QUE score:')
    display(top20.round(4))

## Visualisations

In [ ]:
# ── Score distributions ───────────────────────────────────────────────────────
n_methods = len(scores)
fig, axes = plt.subplots(1, n_methods, figsize=(4 * n_methods, 4), sharey=False)
if n_methods == 1:
    axes = [axes]

for ax, (method, s) in zip(axes, scores.items()):
    if 'is_outlier' in scores_df.columns:
        inlier_scores  = s[scores_df['is_outlier'] == 0]
        outlier_scores = s[scores_df['is_outlier'] == 1]
        ax.hist(inlier_scores,  bins=40, alpha=0.6, label='inliers',  color='steelblue')
        ax.hist(outlier_scores, bins=40, alpha=0.6, label='outliers', color='tomato')
        ax.legend(fontsize=8)
    else:
        ax.hist(s, bins=40, color='steelblue', alpha=0.8)

    ax.set_title(method, fontsize=11)
    ax.set_xlabel('score')
    ax.set_ylabel('count')
    ax.grid(True, alpha=0.3)

plt.suptitle('Score distributions by method', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROCAUC evaluation (only if ground-truth labels are present) ───────────────
def rocauc(inlier_scores: np.ndarray, outlier_scores: np.ndarray) -> float:
    """
    ROCAUC = Pr(score(outlier) > score(inlier)) estimated by Monte Carlo.
    Matches the metric used in the paper.
    """
    n_in, n_out = len(inlier_scores), len(outlier_scores)
    if n_in == 0 or n_out == 0:
        return float('nan')
    chunk = 500
    probs = []
    for i in range(0, n_out, chunk):
        out_chunk = outlier_scores[i:i+chunk]                  # (m,)
        hit = (out_chunk[:, None] > inlier_scores[None, :]).mean(axis=1)  # (m,)
        probs.append(hit.mean())
    return float(np.mean(probs))


if 'is_outlier' in scores_df.columns:
    gt = scores_df['is_outlier'].values
    results = []
    for method, s in scores.items():
        auc = rocauc(s[gt == 0], s[gt == 1])
        results.append({'method': method, 'ROCAUC': round(auc, 4)})

    results_df = pd.DataFrame(results).sort_values('ROCAUC', ascending=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    bar_colors = ['#e04040' if r['method'] == 'que' else '#4a90d9' for _, r in results_df.iterrows()]
    ax.barh(results_df['method'], results_df['ROCAUC'], color=bar_colors)
    ax.set_xlim(0, 1)
    ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.8, label='random baseline')
    for i, (_, row) in enumerate(results_df.iterrows()):
        ax.text(row['ROCAUC'] + 0.01, i, f"{row['ROCAUC']:.3f}", va='center', fontsize=10)
    ax.set_xlabel('ROCAUC  (higher is better)', fontsize=11)
    ax.set_title('Outlier detection performance by method', fontsize=13)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(results_df.to_string(index=False))
else:
    print('No "is_outlier" column found — skipping ROCAUC evaluation.\n'
          'Add a boolean/int column named "is_outlier" (1 = outlier, 0 = inlier) to enable this.')

In [ ]:
# ── Score correlation heatmap ─────────────────────────────────────────────────
if len(score_cols) > 1:
    corr = scores_df[score_cols].corr()
    corr.columns = [c.replace('score_', '') for c in corr.columns]
    corr.index   = [c.replace('score_', '') for c in corr.index]

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, ax=ax, square=True)
    ax.set_title('Pearson correlation between scorer outputs', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── QUE alpha sensitivity ─────────────────────────────────────────────────────
# Shows how ROCAUC changes with alpha (only when ground truth is available).
if 'is_outlier' in scores_df.columns:
    gt = scores_df['is_outlier'].values
    alpha_values = [0, 1, 2, 4, 6, 8, 10, 15, 20]
    alpha_aucs = []

    for a in alpha_values:
        s = que_score(X_t, alpha=float(a)).cpu().numpy()
        auc = rocauc(s[gt == 0], s[gt == 1])
        alpha_aucs.append(auc)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(alpha_values, alpha_aucs, marker='o', color='tomato', linewidth=2, label='QUE')
    ax.axvline(ALPHA, color='grey', linestyle='--', linewidth=0.9, label=f'current α={ALPHA}')
    ax.set_xlabel('Alpha (α)', fontsize=11)
    ax.set_ylabel('ROCAUC', fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_title('QUE ROCAUC vs Alpha', fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Export results

In [ ]:
# ── Save scores to CSV ────────────────────────────────────────────────────────
output_path = 'outlier_scores.csv'
scores_df.to_csv(output_path, index=False)
print(f'Scores saved to {output_path}')
scores_df[score_cols].head(10)